# Numerical Quadrature Laboratory
## From a Faithful VBA Tanh-Sinh Port to Vectorized Quadrature, Custom Gaussian Rules, FFT Methods and Sparse Cubature

#### Gene Boo, 2008, 2012, 2021, 2026

This notebook began as a self-directed study of numerical integration methods in 2008. It incorporated a faithful Python port of the finite-interval `QUAD_TANH_SINH` VBA routine from **Tanh-Sinh Quadrature v5.03 by Graeme Dennes.xlsm** in 2012, followed by further revisions in 2012, 2021 and 2026. It has since developed into a broader laboratory for quadrature, Gaussian-rule construction, performance optimization, Fourier methods and multidimensional cubature, where the other implementations were self-developed and/or adaptations from texts such as Numerical Methods in C.

The fundamental one-dimensional problem is to approximate

$$
I=\int_a^b f(x)\,dx
$$

by a finite weighted sum

$$
I_n=\sum_{i=1}^{n} w_i f(x_i),
$$

where $x_i$ are the quadrature abscissas and $w_i$ are the corresponding weights. The choice of nodes, weights, transformations and stopping criteria determines the accuracy, stability, computational cost and suitability of a method for smooth, singular, oscillatory, weighted, periodic or multidimensional integrands.

---

## Tanh-Sinh implementations

The original finite-interval routine is implemented in two principal forms:

1. **Faithful scalar port**: preserves the VBA refinement order, numerical constants, endpoint protections, transformed abscissas, M&M-style non-overflowing weight formula, convergence logic, result fields and evaluation counting.

2. **NumPy level-batched derivative**: preserves the Tanh-Sinh transformation and nested-node refinement structure while evaluating candidate nodes in NumPy batches. It is intended for vectorized integrands and large batched workloads, but is not presented as an instruction-for-instruction reproduction of the VBA execution path.

For a finite interval $[a,b]$, the commonly printed Tanh-Sinh transformation is

$$
x(t)=\frac{a+b}{2}+\frac{b-a}{2}\tanh\!\left(\frac{\pi}{2}\sinh t\right),
$$

with derivative

$$
\frac{dx}{dt}=\frac{b-a}{2}\,
\frac{\frac{\pi}{2}\cosh t}
{\cosh^2\!\left(\frac{\pi}{2}\sinh t\right)}.
$$

Thus,

$$
\int_a^b f(x)\,dx
=
\int_{-\infty}^{\infty} f\!\bigl(x(t)\bigr)\frac{dx}{dt}\,dt
\approx
h\sum_{k=-N}^{N}f\!\bigl(x(kh)\bigr)\frac{dx}{dt}(kh).
$$

Graeme Dennes's workbook uses an algebraically related double-exponential mapping together with a stable non-overflowing weight formulation. The faithful Python routine retains the workbook's actual formulas rather than silently replacing them with the conventional $\pi/2$-scaled presentation above.

The workbook's expression-string and Excel evaluation mechanism is intentionally replaced by a Python callable `f(x)`. This separates the numerical algorithms from Excel formula parsing and makes the computational core suitable for Jupyter, ExcelBridge, Python services and future native C++ DLL implementations.

---

## Implemented methods

The notebook implements, validates and benchmarks:

- adaptive Simpson quadrature;
- Clenshaw-Curtis quadrature;
- Gauss-Legendre, Gauss-Hermite, Gauss-Laguerre and Gauss-Jacobi quadrature;
- Tanh-Sinh and double-exponential quadrature;
- NumPy-vectorized and Numba-compiled implementations;
- Golub-Welsch construction of custom Gaussian weights and abscissas;
- Gaussian rules for positive continuous densities and discrete measures;
- SciPy adaptive, fixed-order and Tanh-Sinh integration, where available;
- direct periodic integration and FFT-based convolution;
- tensor-product cubature in two, three and four dimensions;
- Smolyak sparse-grid cubature.

An $n$-point Gaussian rule has the form

$$
\int_{\Omega} f(x)w(x)\,dx
\approx
\sum_{i=1}^{n}w_i f(x_i),
$$

and is exact for polynomials through degree

$$
2n-1
$$

under its defining weight function.

Golub-Welsch obtains the abscissas from the eigenvalues of a symmetric tridiagonal Jacobi matrix,

$$
J_n=
\begin{bmatrix}
\alpha_0 & \beta_1 & 0 & \cdots & 0\\
\beta_1 & \alpha_1 & \beta_2 & \ddots & \vdots\\
0 & \beta_2 & \alpha_2 & \ddots & 0\\
\vdots & \ddots & \ddots & \ddots & \beta_{n-1}\\
0 & \cdots & 0 & \beta_{n-1} & \alpha_{n-1}
\end{bmatrix},
$$

and calculates the weights from the first components of its normalized eigenvectors:

$$
w_i=\mu_0\left(v_{1i}\right)^2,
\qquad
\mu_0=\int d\mu(x).
$$

The notebook compares 48-point and 128-point custom Golub-Welsch rules to demonstrate convergence and the trade-off between construction cost and repeated application speed. For a precomputed rule applied $M$ times,

$$
T_{\mathrm{first}}=T_{\mathrm{setup}}+T_{\mathrm{apply}},
$$

while the amortized cost per application is

$$
T_{\mathrm{amortized}}
=
\frac{T_{\mathrm{setup}}}{M}+T_{\mathrm{apply}}.
$$

---

## Fourier methods and cubature

For a periodic function sampled uniformly on an interval of length $L$,

$$
\int_0^L f(x)\,dx
\approx
\frac{L}{N}\sum_{j=0}^{N-1}f(x_j)
=
\frac{L}{N}\widehat{f}_0,
$$

where $\widehat{f}_0$ is the zero-frequency Fourier coefficient. FFT methods become particularly useful when the complete spectrum or a full periodic convolution is required:

$$
\widehat{h}_k=\widehat{f}_k\widehat{g}_k,
\qquad
h(x)=\int_0^L f(t)g(x-t)\,dt.
$$

Cubature generalizes quadrature to $d$ dimensions:

$$
I_d=\int_{\Omega\subset\mathbb{R}^d}f(\mathbf{x})\,d\mathbf{x}.
$$

A tensor-product rule is

$$
I_d\approx
\sum_{i_1=1}^{n_1}\cdots\sum_{i_d=1}^{n_d}
\left(\prod_{r=1}^{d}w_{i_r}^{(r)}\right)
f\!\left(x_{i_1}^{(1)},\ldots,x_{i_d}^{(d)}\right).
$$

Using the same $n$-point rule in every dimension requires

$$
N_{\mathrm{tensor}}=n^d
$$

function evaluations. The Smolyak sparse-grid construction reduces this growth by combining nested lower-order tensor rules.

---

## Scope and validation philosophy

The examples span physics, chemistry, biology, finance, pure mathematics and applied mathematics. They include endpoint singularities, oscillatory integrals, weighted infinite-domain integrals, non-standard probability mixtures, option payoffs, discrete expectations, integral operators, periodic convolution and multidimensional cubature.

Accuracy, convergence and speed are evaluated together. Where applicable, the notebook reports the absolute error

$$
E_{\mathrm{abs}}=\left|I_n-I_{\mathrm{ref}}\right|,
$$

relative error

$$
E_{\mathrm{rel}}
=
\frac{\left|I_n-I_{\mathrm{ref}}\right|}
{\left|I_{\mathrm{ref}}\right|},
$$

function-evaluation counts, rule-construction time, repeated application time, first-use time, and serial, vectorized, JIT-compiled and parallel timings.

No method is assumed to be universally optimal. Each technique is assessed according to the smoothness, singularity structure, integration domain, weighting measure, dimensionality, periodicity and batch structure of the problem.

The notebook is intended as both a technical reference and an experimental foundation for high-performance numerical functions exposed through ExcelBridge, Python services and future native C++ libraries.


In [1]:
from __future__ import annotations
from dataclasses import dataclass
from time import perf_counter
from typing import Callable
import math
import numpy as np

@dataclass(frozen=True)
class QuadResult:
    value: float
    error: float
    evaluations: int
    elapsed_seconds: float
    levels: int
    converged: bool


## 1. Faithful scalar port

Graeme Dennes's VBA constants are preserved exactly:

$$
\varepsilon=10^{-15},
\qquad
\mathrm{TOL}=10^{-7},
\qquad
k_{\max}=6.
$$

At refinement level $k$ the step size is

$$
h_k=2^{-k}.
$$

At level zero every positive transformed node is used; later levels add only odd-indexed nodes, matching `J = J + 2` in the workbook. If `ss` denotes the accumulated transformed sum, the workbook returns

$$
I_k=ss\,\frac{b-a}{2}\,h_k.
$$


In [2]:
def quad_tanh_sinh_faithful(
    f: Callable[[float], float],
    a: float,
    b: float,
    *,
    eps: float = 1e-15,
    tol: float = 1e-7,
    maxlevel: int = 6,
) -> QuadResult:
    """Faithful scalar translation of VBA QUAD_TANH_SINH for finite [a,b]."""
    if not callable(f):
        raise TypeError("f must be callable")
    if not (math.isfinite(a) and math.isfinite(b)):
        raise ValueError("finite limits are required")
    if b < a:
        raise ValueError("upper limit must be >= lower limit")
    if a == b:
        return QuadResult(0.0, 0.0, 0, 0.0, 0, True)

    started = perf_counter()
    bma2 = (b - a) / 2.0
    bpa2 = (b + a) / 2.0
    k = 0
    ss = float(f(bpa2))
    evals = 1
    sslast = 0.0
    errval = math.inf
    h = 1.0

    while True:
        ssp = 0.0
        j = 1
        h = 2.0 ** (-k)
        fp = 0.0  # VBA variables persist; normally both are overwritten at each valid node.
        fm = 0.0

        while True:
            t = h * j
            expt = math.exp(t)
            u = math.exp(1.0 / expt - expt)
            r = 2.0 * u / (1.0 + u)  # 1 - tanh(sinh(t)), evaluated stably
            if r == 0.0 or r == 1.0:
                break

            x = bma2 * r
            left = a + x
            right = b - x
            if left > a:
                fp = float(f(left)); evals += 1
            if right < b:
                fm = float(f(right)); evals += 1

            w = (expt + 1.0 / expt) * r / (1.0 + u)
            p = (fp + fm) * w
            ssp += p
            j += 2 if k > 0 else 1

            if not (abs(p) > abs(ssp * eps)):
                break

        ss += ssp
        errval = abs(2.0 * sslast / ss - 1.0) if ss != 0.0 else math.inf
        sslast = ss
        k += 1
        if not (errval >= tol and k <= maxlevel):
            break

    return QuadResult(
        value=ss * bma2 * h,
        error=errval,
        evaluations=evals,
        elapsed_seconds=perf_counter() - started,
        levels=k,
        converged=errval < tol,
    )


## 2. NumPy level-batched Tanh-Sinh derivative

This derivative preserves the workbook transform, refinement levels and stopping formula, while evaluating candidate nodes in NumPy arrays. It is **not presented as an instruction-for-instruction VBA equivalent**: it evaluates candidate points speculatively in batches, discards the unused tail after the stopping point, and reports logical quadrature evaluations rather than physical calls made inside the vectorized function. The faithful scalar routine remains the audit reference.

For genuine speed, the integrand should accept NumPy arrays. Small arrays can still be slower than scalar Python because allocation and masking dominate.


In [3]:
def quad_tanh_sinh_vectorized(
    f: Callable[[np.ndarray], np.ndarray],
    a: float,
    b: float,
    *,
    eps: float = 1e-15,
    tol: float = 1e-7,
    maxlevel: int = 6,
    node_cap: int = 4096,
) -> QuadResult:
    """NumPy-batched equivalent of the workbook's finite-interval algorithm."""
    if b < a:
        raise ValueError("upper limit must be >= lower limit")
    if a == b:
        return QuadResult(0.0, 0.0, 0, 0.0, 0, True)
    if node_cap < 8:
        raise ValueError("node_cap must be at least 8")

    started = perf_counter()
    bma2 = (b - a) / 2.0
    bpa2 = (b + a) / 2.0
    centre = np.asarray(f(np.array([bpa2], dtype=np.float64)), dtype=np.float64).reshape(-1)
    if centre.size != 1:
        raise ValueError("f must return one value per input point")
    ss = float(centre[0])
    evals = 1
    sslast = 0.0
    errval = math.inf
    h = 1.0
    k = 0

    while True:
        h = 2.0 ** (-k)
        step = 2 if k > 0 else 1
        j = np.arange(1, 1 + step * node_cap, step, dtype=np.float64)
        t = h * j

        # exp(t) overflows only far beyond the useful range. Mask before exponentiation.
        valid_t = t < 7.0
        t = t[valid_t]
        expt = np.exp(t)
        u = np.exp(1.0 / expt - expt)
        r = 2.0 * u / (1.0 + u)
        valid = (r != 0.0) & (r != 1.0)
        r, expt, u = r[valid], expt[valid], u[valid]
        if r.size == 0:
            break

        x = bma2 * r
        left = a + x
        right = b - x
        endpoint_ok = (left > a) & (right < b)
        left, right = left[endpoint_ok], right[endpoint_ok]
        expt, u, r = expt[endpoint_ok], u[endpoint_ok], r[endpoint_ok]
        if r.size == 0:
            break

        pts = np.concatenate((left, right))
        vals = np.asarray(f(pts), dtype=np.float64).reshape(-1)
        if vals.size != pts.size:
            raise ValueError("f must return one value per input point")
        n = r.size
        fp, fm = vals[:n], vals[n:]
        evals += 2 * n
        w = (expt + 1.0 / expt) * r / (1.0 + u)
        p = (fp + fm) * w
        prefix = np.cumsum(p)
        stop = np.flatnonzero(np.abs(p) <= np.abs(prefix * eps))
        used = int(stop[0] + 1) if stop.size else p.size
        ssp = float(prefix[used - 1])
        # Correct the provisional batch count to count only nodes actually used.
        evals -= 2 * (n - used)

        ss += ssp
        errval = abs(2.0 * sslast / ss - 1.0) if ss != 0.0 else math.inf
        sslast = ss
        k += 1
        if not (errval >= tol and k <= maxlevel):
            break

    return QuadResult(
        value=ss * bma2 * h,
        error=errval,
        evaluations=evals,
        elapsed_seconds=perf_counter() - started,
        levels=k,
        converged=errval < tol,
    )


## 3. Numerical validation

The cases include smooth, endpoint-singular, sharply peaked, and logarithmically singular functions. The expected values are closed form, so no external package is needed as the reference.


In [4]:
tests = [
    ("polynomial", lambda x: x**4 - 2*x + 1, lambda x: x**4 - 2*x + 1, 0.0, 2.0, 22/5),
    ("sqrt endpoint", math.sqrt, np.sqrt, 0.0, 1.0, 2/3),
    ("inverse sqrt", lambda x: 1/math.sqrt(x), lambda x: 1/np.sqrt(x), 0.0, 1.0, 2.0),
    ("log endpoint", math.log, np.log, 0.0, 1.0, -1.0),
    ("Cauchy peak", lambda x: 1/(1+x*x), lambda x: 1/(1+x*x), -10.0, 10.0, 2*math.atan(10.0)),
]

rows = []
for name, fs, fv, a, b, expected in tests:
    scalar = quad_tanh_sinh_faithful(fs, a, b)
    vector = quad_tanh_sinh_vectorized(fv, a, b)
    rows.append((name, expected, scalar.value, abs(scalar.value-expected), vector.value, abs(vector.value-expected), scalar.evaluations, vector.evaluations))

header = ("case", "expected", "faithful", "abs err", "vectorized", "abs err", "evals S", "evals V")
print(" | ".join(f"{h:>14}" for h in header))
print("-" * 137)
for row in rows:
    print(f"{row[0]:>14} | {row[1]:14.10g} | {row[2]:14.10g} | {row[3]:14.3e} | {row[4]:14.10g} | {row[5]:14.3e} | {row[6]:14d} | {row[7]:14d}")

assert max(r[3] for r in rows) < 2e-7
assert max(r[5] for r in rows) < 2e-7


          case |       expected |       faithful |        abs err |     vectorized |        abs err |        evals S |        evals V
-----------------------------------------------------------------------------------------------------------------------------------------
    polynomial |            4.4 |            4.4 |      0.000e+00 |            4.4 |      7.105e-15 |             62 |             57
 sqrt endpoint |   0.6666666667 |   0.6666666667 |      0.000e+00 |   0.6666666667 |      2.220e-16 |             61 |             57
  inverse sqrt |              2 |              2 |      8.882e-16 |    1.999999994 |      6.308e-09 |             35 |             29
  log endpoint |             -1 |             -1 |      0.000e+00 |             -1 |      9.104e-15 |             62 |             57
   Cauchy peak |    2.942255349 |    2.942255349 |      8.882e-16 |    2.942255349 |      8.882e-16 |            455 |            455


# Part II. Performance-oriented Tanh-Sinh implementations

The first NumPy version batches the nodes of one small integral. That is not automatically fast because allocation, masking, concatenation and prefix-sum overhead can exceed a short scalar loop. This section adds:

- a **lean scalar implementation** with fewer objects and branches;
- a **Numba-compiled scalar implementation**;
- a **Numba parallel batch implementation** using `prange`;
- workload examples where vectorization has enough arithmetic intensity or batch size to win.

The JIT functions below use an integer integrand identifier so Numba can compile the complete hot path. For ExcelBridge, this maps naturally to a registered-integrand catalogue or generated specialist kernels.


In [5]:
from scipy import integrate, special
from numpy.polynomial.legendre import leggauss
from numpy.polynomial.hermite import hermgauss
from numba import njit, prange, set_num_threads, get_num_threads
import os

# Registered benchmark kernels shared by scalar, vectorized and Numba paths.
# 0: exp(-p*x^2)*cos(q*x); 1: 1/sqrt(x+p); 2: log(x+p); 3: expensive smooth kernel

def kernel_scalar(x, kind=0, p=1.0, q=7.0):
    if kind == 0: return math.exp(-p*x*x) * math.cos(q*x)
    if kind == 1: return 1.0 / math.sqrt(x+p)
    if kind == 2: return math.log(x+p)
    y=x
    for _ in range(64):
        y = math.sin(y+p) + math.cos(q*y)/(1.0+y*y)
    return math.exp(-x*x) * y

def kernel_numpy(x, kind=0, p=1.0, q=7.0):
    x=np.asarray(x)
    if kind == 0: return np.exp(-p*x*x) * np.cos(q*x)
    if kind == 1: return 1.0 / np.sqrt(x+p)
    if kind == 2: return np.log(x+p)
    y=x.copy()
    for _ in range(64):
        y = np.sin(y+p) + np.cos(q*y)/(1.0+y*y)
    return np.exp(-x*x) * y

@njit(cache=False, fastmath=False)
def kernel_numba(x, kind, p, q):
    if kind == 0: return math.exp(-p*x*x) * math.cos(q*x)
    if kind == 1: return 1.0 / math.sqrt(x+p)
    if kind == 2: return math.log(x+p)
    y=x
    for _ in range(64):
        y = math.sin(y+p) + math.cos(q*y)/(1.0+y*y)
    return math.exp(-x*x) * y


In [6]:
def tanh_sinh_fast_scalar(kind, p, q, a, b, eps=1e-15, tol=1e-7, maxlevel=6):
    """Lean scalar version of the workbook algorithm; returns only the numerical result."""
    half=(b-a)*0.5; mid=(a+b)*0.5
    ss=kernel_scalar(mid,kind,p,q); prev=0.0; h=1.0
    for k in range(maxlevel+1):
        h=2.0**(-k); subtotal=0.0; j=1; inc=2 if k else 1
        while True:
            t=h*j; e=math.exp(t); u=math.exp(1.0/e-e); r=2.0*u/(1.0+u)
            if r==0.0 or r==1.0: break
            x=half*r; xl=a+x; xr=b-x
            if xl<=a or xr>=b: break
            w=(e+1.0/e)*r/(1.0+u)
            term=(kernel_scalar(xl,kind,p,q)+kernel_scalar(xr,kind,p,q))*w
            subtotal += term
            if abs(term) <= abs(subtotal*eps): break
            j += inc
        ss += subtotal
        err=abs(2.0*prev/ss-1.0) if ss else math.inf
        prev=ss
        if err < tol: break
    return ss*half*h

@njit(cache=False, fastmath=False)
def tanh_sinh_numba(kind, p, q, a, b, eps=1e-15, tol=1e-7, maxlevel=6):
    half=(b-a)*0.5; mid=(a+b)*0.5
    ss=kernel_numba(mid,kind,p,q); prev=0.0; h=1.0
    for k in range(maxlevel+1):
        h=2.0**(-k); subtotal=0.0; j=1; inc=2 if k else 1
        while True:
            t=h*j; e=math.exp(t); u=math.exp(1.0/e-e); r=2.0*u/(1.0+u)
            if r==0.0 or r==1.0: break
            x=half*r; xl=a+x; xr=b-x
            if xl<=a or xr>=b: break
            w=(e+1.0/e)*r/(1.0+u)
            term=(kernel_numba(xl,kind,p,q)+kernel_numba(xr,kind,p,q))*w
            subtotal += term
            if abs(term) <= abs(subtotal*eps): break
            j += inc
        ss += subtotal
        err=abs(2.0*prev/ss-1.0) if ss else 1e300
        prev=ss
        if err < tol: break
    return ss*half*h

@njit(cache=False, fastmath=False)
def tanh_sinh_numba_batch_serial(kind, ps, qs, a, b):
    out=np.empty(ps.size)
    for i in range(ps.size):
        out[i]=tanh_sinh_numba(kind,ps[i],qs[i],a,b)
    return out

@njit(parallel=True, cache=False, fastmath=False)
def tanh_sinh_numba_parallel(kind, ps, qs, a, b):
    out=np.empty(ps.size)
    for i in prange(ps.size):
        out[i]=tanh_sinh_numba(kind,ps[i],qs[i],a,b)
    return out

# Warm up compilation once; exclude JIT startup from timed measurements.
_ = tanh_sinh_numba(0,1.0,7.0,-3.0,3.0)
_ = tanh_sinh_numba_batch_serial(0,np.ones(4),np.full(4,7.0),-3.0,3.0)
_ = tanh_sinh_numba_parallel(0,np.ones(4),np.full(4,7.0),-3.0,3.0)
print('Numba threads:', get_num_threads())


Numba threads: 18


## When vectorization can win

Three separate cases are benchmarked:

1. **One cheap integral**: scalar loops often win.
2. **One expensive vectorizable integrand**: NumPy has enough arithmetic work per batch to recover its setup cost.
3. **Many independent integrals**: two-dimensional broadcasting or Numba `prange` amortizes overhead and exposes coarse-grained parallelism.

The comparison reports ratios explicitly. A ratio greater than one means the method named in the numerator took longer.


In [7]:
def median_time(fn, repeats=7):
    times=[]; ans=None
    for _ in range(repeats):
        t0=perf_counter(); ans=fn(); times.append(perf_counter()-t0)
    return float(np.median(times)), ans

# Case A: one cheap integral, repeated internally to produce stable timings.
def repeat_call(fn, n):
    ans=None
    for _ in range(n): ans=fn()
    return ans

N=1000
cheap_scalar_t, cheap_scalar = median_time(lambda: repeat_call(lambda: tanh_sinh_fast_scalar(0,1.0,7.0,-3,3),N))
cheap_vector_t, cheap_vector = median_time(lambda: repeat_call(lambda: quad_tanh_sinh_vectorized(lambda x: kernel_numpy(x,0,1,7),-3,3).value,N))
cheap_numba_t, cheap_numba = median_time(lambda: repeat_call(lambda: tanh_sinh_numba(0,1.0,7.0,-3,3),N))

# Case B: one expensive integrand.
M=100
exp_scalar_t, exp_scalar = median_time(lambda: repeat_call(lambda: tanh_sinh_fast_scalar(3,0.7,3.2,-3,3),M))
exp_vector_t, exp_vector = median_time(lambda: repeat_call(lambda: quad_tanh_sinh_vectorized(lambda x: kernel_numpy(x,3,0.7,3.2),-3,3).value,M))
exp_numba_t, exp_numba = median_time(lambda: repeat_call(lambda: tanh_sinh_numba(3,0.7,3.2,-3,3),M))

print(f'Cheap x{N}: scalar={cheap_scalar_t:.4f}s, NumPy={cheap_vector_t:.4f}s, Numba={cheap_numba_t:.4f}s')
print(f'  NumPy/scalar={cheap_vector_t/cheap_scalar_t:.2f}x; scalar/Numba={cheap_scalar_t/cheap_numba_t:.2f}x')
print(f'Expensive x{M}: scalar={exp_scalar_t:.4f}s, NumPy={exp_vector_t:.4f}s, Numba={exp_numba_t:.4f}s')
print(f'  NumPy/scalar={exp_vector_t/exp_scalar_t:.2f}x; scalar/Numba={exp_scalar_t/exp_numba_t:.2f}x')
print('Agreement:', abs(cheap_scalar-cheap_vector), abs(cheap_scalar-cheap_numba), abs(exp_scalar-exp_vector), abs(exp_scalar-exp_numba))


Cheap x1000: scalar=0.1149s, NumPy=0.2738s, Numba=0.0438s
  NumPy/scalar=2.38x; scalar/Numba=2.62x
Expensive x100: scalar=0.6147s, NumPy=0.2822s, Numba=0.0606s
  NumPy/scalar=0.46x; scalar/Numba=10.14x
Agreement: 0.0 0.0 0.0 0.0


In [8]:
# Case C: many independent parameterized integrals.
B=20_000
ps=np.linspace(0.5,2.0,B)
qs=np.linspace(1.0,20.0,B)

serial_t, serial_out = median_time(lambda: tanh_sinh_numba_batch_serial(0,ps,qs,-3,3), repeats=3)
parallel_t, parallel_out = median_time(lambda: tanh_sinh_numba_parallel(0,ps,qs,-3,3), repeats=3)
print(f'Batch size={B:,}; compiled serial={serial_t:.4f}s; Numba parallel={parallel_t:.4f}s')
print(f'Parallel speedup={serial_t/parallel_t:.2f}x on {get_num_threads()} configured threads')
print('Maximum difference:', np.max(np.abs(serial_out-parallel_out)))


Batch size=20,000; compiled serial=0.1642s; Numba parallel=0.0395s
Parallel speedup=4.16x on 18 configured threads
Maximum difference: 0.0


In [9]:
# A workload where NumPy vectorization should clearly win: many fixed-order Gauss-Legendre integrals.
GL_N=64
GL_X,GL_W=leggauss(GL_N)
GL_XX=3.0*GL_X  # interval [-3,3]

def gauss_legendre_batch_pq(ps, qs):
    values=np.exp(-ps[:,None]*GL_XX[None,:]**2)*np.cos(qs[:,None]*GL_XX[None,:])
    return 3.0*(values @ GL_W)

def gauss_legendre_one_pq(p,q):
    return 3.0*np.dot(GL_W,np.exp(-p*GL_XX*GL_XX)*np.cos(q*GL_XX))

VB=10_000
vps=np.linspace(0.5,2.0,VB); vqs=np.linspace(1.0,20.0,VB)
vec_t,vec_out=median_time(lambda: gauss_legendre_batch_pq(vps,vqs),repeats=5)
loop_t,loop_out=median_time(lambda: np.array([gauss_legendre_one_pq(vps[i],vqs[i]) for i in range(VB)]),repeats=3)
print(f'Batched Gauss-Legendre, {VB:,} integrals: NumPy matrix={vec_t:.4f}s, Python loop={loop_t:.4f}s')
print(f'Vectorization speedup={loop_t/vec_t:.2f}x; max difference={np.max(np.abs(vec_out-loop_out)):.3e}')


Batched Gauss-Legendre, 10,000 integrals: NumPy matrix=0.0144s, Python loop=0.0461s
Vectorization speedup=3.20x; max difference=6.661e-16


# Part III. Additional quadrature families

The implementations are grouped by mathematical role:

- **Adaptive Newton-Cotes**: adaptive Simpson quadrature on finite intervals.
- **Gaussian quadrature**: fixed-order rules tailored to a weight function.
- **Cosine-grid spectral quadrature**: Clenshaw-Curtis on Chebyshev extrema.
- **Double exponential**: the workbook-faithful Tanh-Sinh method.

Gauss-Hermite is not a drop-in replacement for finite-interval quadrature. It approximates

$$
\int_{-\infty}^{\infty}e^{-x^2}f(x)\,dx
\approx
\sum_{i=1}^{n}w_i f(x_i).
$$


## A. Adaptive Simpson


In [10]:
def adaptive_simpson(f, a, b, tol=1e-12, max_depth=20):
    fa,fb=f(a),f(b); m=(a+b)/2; fm=f(m)
    whole=(b-a)*(fa+4*fm+fb)/6
    evals=3
    def rec(a,b,fa,fm,fb,whole,tol,depth):
        nonlocal evals
        m=(a+b)/2; lm=(a+m)/2; rm=(m+b)/2
        flm,frm=f(lm),f(rm); evals += 2
        left=(m-a)*(fa+4*flm+fm)/6
        right=(b-m)*(fm+4*frm+fb)/6
        delta=left+right-whole
        if depth<=0 or abs(delta)<=15*tol:
            return left+right+delta/15
        return rec(a,m,fa,flm,fm,left,tol/2,depth-1)+rec(m,b,fm,frm,fb,right,tol/2,depth-1)
    return rec(a,b,fa,fm,fb,whole,tol,max_depth), evals


## B. Gauss-Legendre and Gauss-Hermite


In [11]:
def gauss_legendre(f, a, b, n=64):
    x,w=leggauss(n); mid=(a+b)/2; half=(b-a)/2
    return half*np.dot(w, f(mid+half*x))

def gauss_hermite_weighted(f, n=64):
    # Approximates integral exp(-x^2) f(x) dx over R.
    x,w=hermgauss(n)
    return np.dot(w,f(x))


## C. Clenshaw-Curtis

Weights are generated by the stable cosine-series formula used for Chebyshev extrema. This is an `O(n^2)` setup implementation for clarity; cache the nodes and weights when repeatedly integrating at a fixed order.


In [12]:
from functools import lru_cache

@lru_cache(maxsize=32)
def clenshaw_curtis_rule(n):
    if n < 2: raise ValueError('n must be at least 2')
    theta=np.pi*np.arange(n+1)/n
    x=np.cos(theta); w=np.zeros(n+1)
    ii=np.arange(1,n); v=np.ones(n-1)
    if n%2==0:
        w[0]=w[n]=1/(n*n-1)
        for k in range(1,n//2): v -= 2*np.cos(2*k*theta[ii])/(4*k*k-1)
        v -= np.cos(n*theta[ii])/(n*n-1)
    else:
        w[0]=w[n]=1/(n*n)
        for k in range(1,(n+1)//2): v -= 2*np.cos(2*k*theta[ii])/(4*k*k-1)
    w[ii]=2*v/n
    return x,w

def clenshaw_curtis(f,a,b,n=128):
    x,w=clenshaw_curtis_rule(n); mid=(a+b)/2; half=(b-a)/2
    return half*np.dot(w,f(mid+half*x))


# Part IV. Correctness and timing comparison against SciPy and NumPy building blocks

The fixed-order self-written methods use NumPy's `leggauss` and `hermgauss` only to generate canonical Gaussian nodes and weights. SciPy comparisons use:

- `scipy.integrate.quad` (QUADPACK adaptive integration),
- `scipy.integrate.fixed_quad` (fixed-order Gaussian quadrature),
- `scipy.integrate.tanhsinh` when available in the installed SciPy version.

Timings exclude Numba's first compilation and use median wall-clock time. Results should be interpreted together with absolute error, because a faster method is not comparable if it misses the requested accuracy.


## Endpoint-singularity handling and fair setup timing

Closed Clenshaw-Curtis and ordinary adaptive Simpson evaluate interval endpoints. Therefore, applying them directly to $\log x$ on $[0,1]$ forces an evaluation at $x=0$. This is an incompatible rule-function pairing rather than a Python defect.

For a left-endpoint singularity, the notebook uses

$$
x=a+(b-a)t^2,
\qquad 0\le t\le1,
$$

with Jacobian

$$
dx=2(b-a)t\,dt.
$$

Therefore,

$$
\int_a^b f(x)\,dx
=
\int_0^1 f\!\left(a+(b-a)t^2\right)2(b-a)t\,dt.
$$

Gauss-Legendre nodes and weights are cached so repeated timings measure rule application rather than repeated rule construction. Direct Clenshaw-Curtis is retained to demonstrate the endpoint failure mode, with warnings handled locally and an explicit status reported.


In [13]:
# Singular-aware endpoint transform x = a + (b-a)t^2.
# It removes the direct endpoint evaluation at x=a and is useful for algebraic/log singularities.
def left_endpoint_transform_scalar(f, a, b):
    span=b-a
    def g(t):
        if t == 0.0:
            # For the cases below, the transformed limit is zero.
            return 0.0
        x=a+span*t*t
        return f(x)*(2.0*span*t)
    return g

def left_endpoint_transform_vector(f, a, b):
    span=b-a
    def g(t):
        t=np.asarray(t,dtype=float)
        x=a+span*t*t
        out=np.zeros_like(t)
        mask=t!=0.0
        out[mask]=f(x[mask])*(2.0*span*t[mask])
        return out
    return g

@lru_cache(maxsize=32)
def cached_legendre_rule(n):
    return leggauss(n)

def gauss_legendre_cached(f,a,b,n=128):
    x,w=cached_legendre_rule(n); mid=(a+b)/2; half=(b-a)/2
    return half*np.dot(w,f(mid+half*x))


In [14]:
def scipy_tanhsinh_value(f,a,b):
    if not hasattr(integrate,'tanhsinh'):
        return np.nan
    return float(integrate.tanhsinh(f,a,b).integral)

cases=[
 ('smooth', lambda x: np.exp(-x*x)*np.cos(7*x), lambda x: math.exp(-x*x)*math.cos(7*x), -3.,3.),
 ('endpoint sqrt', np.sqrt, math.sqrt, 0.,1.),
 ('log singular', np.log, math.log, 0.,1.),
 ('sharp peak', lambda x: 1/(1+1000*x*x), lambda x: 1/(1+1000*x*x), -1.,1.),
]

def reference_value(fv,a,b):
    # Tight QUADPACK reference; split endpoint-singular intervals are still handled by quad.
    return integrate.quad(lambda x: float(fv(x)),a,b,epsabs=1e-14,epsrel=1e-14,limit=1000)[0]

def compare_case(name,fv,fs,a,b,repeats=31):
    ref=reference_value(fv,a,b)
    left_singular = name in {'endpoint sqrt','log singular'}
    methods={
      'Tanh-Sinh faithful': lambda: quad_tanh_sinh_faithful(fs,a,b).value,
      'Tanh-Sinh NumPy': lambda: quad_tanh_sinh_vectorized(fv,a,b).value,
      'Gauss-Legendre 128 cached': lambda: gauss_legendre_cached(fv,a,b,128),
      'Clenshaw-Curtis 256': lambda: clenshaw_curtis(fv,a,b,256),
      'SciPy quad': lambda: integrate.quad(lambda x: float(fv(x)),a,b,limit=1000)[0],
      'SciPy fixed_quad 128': lambda: integrate.fixed_quad(fv,a,b,n=128)[0],
    }
    if left_singular:
        gs=left_endpoint_transform_scalar(fs,a,b)
        gv=left_endpoint_transform_vector(fv,a,b)
        methods['Adaptive Simpson transformed']=lambda: adaptive_simpson(gs,0.0,1.0,1e-11)[0]
        methods['Clenshaw-Curtis transformed']=lambda: clenshaw_curtis(gv,0.0,1.0,256)
        methods['Gauss-Legendre transformed']=lambda: gauss_legendre_cached(gv,0.0,1.0,128)
    else:
        methods['Adaptive Simpson']=lambda: adaptive_simpson(fs,a,b,1e-11)[0]
    if hasattr(integrate,'tanhsinh'):
        methods['SciPy tanhsinh']=lambda: scipy_tanhsinh_value(fv,a,b)
    out=[]
    with np.errstate(divide='ignore',invalid='ignore',over='ignore'):
        for label,fn in methods.items():
            try:
                t,val=median_time(fn,repeats)
                val=float(val)
                status='ok' if np.isfinite(val) else 'non-finite endpoint result'
                out.append((name,label,val,abs(val-ref),t*1e6,status))
            except Exception as e:
                out.append((name,label,np.nan,np.nan,np.nan,f'{type(e).__name__}: {e}'))
    return ref,out

all_rows=[]
for case in cases:
    ref,rows=compare_case(*case)
    all_rows.extend(rows)
    print(f'\n{case[0]} reference={ref:.16g}')
    for _,label,val,err,us,status in rows:
        print(f'{label:31s} value={val: .12g} abs_err={err:.3e} median={us:9.2f} us  [{status}]')



smooth reference=3.500121461344011e-05
Tanh-Sinh faithful              value= 3.50012146141e-05 abs_err=6.317e-16 median=    98.60 us  [ok]
Tanh-Sinh NumPy                 value= 3.50012146141e-05 abs_err=6.317e-16 median=   230.10 us  [ok]
Gauss-Legendre 128 cached       value= 3.50012146134e-05 abs_err=2.995e-17 median=    10.10 us  [ok]
Clenshaw-Curtis 256             value= 3.50012146135e-05 abs_err=2.469e-17 median=    11.60 us  [ok]
SciPy quad                      value= 3.50012146137e-05 abs_err=2.233e-16 median=   175.60 us  [ok]
SciPy fixed_quad 128            value= 3.50012146141e-05 abs_err=6.180e-16 median=    12.30 us  [ok]
Adaptive Simpson                value= 3.50012146129e-05 abs_err=5.053e-16 median=  4312.90 us  [ok]

endpoint sqrt reference=0.6666666666666669
Tanh-Sinh faithful              value= 0.666666666667 abs_err=2.220e-16 median=    28.20 us  [ok]
Tanh-Sinh NumPy                 value= 0.666666666667 abs_err=4.441e-16 median=   244.40 us  [ok]
Gauss-Legendr

## Gauss-Hermite-specific comparison

The benchmark uses the correctly weighted identity

$$
\int_{-\infty}^{\infty}e^{-x^2}\cos(qx)\,dx
=
\sqrt{\pi}\,e^{-q^2/4}.
$$


In [15]:
q=3.0
expected=math.sqrt(math.pi)*math.exp(-q*q/4)
for n in (8,16,32,64):
    t,val=median_time(lambda n=n: gauss_hermite_weighted(lambda x: np.cos(q*x),n),repeats=101)
    print(f'n={n:2d}: value={val:.16g}, abs_err={abs(val-expected):.3e}, median={t*1e6:.2f} us')
quad_val=integrate.quad(lambda x: math.exp(-x*x)*math.cos(q*x),-np.inf,np.inf)[0]
print('SciPy quad on (-inf,inf):',quad_val,'abs_err=',abs(quad_val-expected))


n= 8: value=0.1866187196884784, abs_err=1.965e-04, median=115.40 us
n=16: value=0.186815261455799, abs_err=1.333e-12, median=203.10 us
n=32: value=0.186815261457132, abs_err=3.331e-16, median=616.70 us
n=64: value=0.186815261457132, abs_err=3.053e-16, median=953.40 us
SciPy quad on (-inf,inf): 0.18681526145713176 abs_err= 8.326672684688674e-17


# Part V. Parallelism and C++ direction

## Where parallelism belongs

Parallelize **across independent integrals**, scenarios, instruments or parameter rows. Avoid parallelizing the 20-200 nodes inside one adaptive integral: synchronization and reduction overhead usually exceed the work, and adaptive branches cause load imbalance.

Recommended progression for ExcelBridge:

1. **One integral**: Numba-specialized scalar or native C++ scalar kernel.
2. **Many integrals in Python**: Numba `prange`, with data in contiguous NumPy arrays.
3. **Native Windows DLL**: C++20 `std::execution::par`, OpenMP `#pragma omp parallel for`, Intel oneTBB `parallel_for`, or Microsoft PPL `concurrency::parallel_for`.
4. **Very large fixed grids**: SIMD inside each worker with compiler auto-vectorization, `std::experimental::simd`/C++26 `std::simd`, or explicit AVX2/AVX-512 only after profiling.

## C++ options

- **Microsoft PPL `concurrency::parallel_for`** is suitable for your MSVC/Windows x64 and Win32 builds and has low integration friction, but it is Microsoft-specific.
- **OpenMP** is the easiest portable annotation model and works with MSVC via `/openmp`; check which OpenMP feature level your compiler actually supports.
- **oneTBB** offers work-stealing and better control for irregular workloads, at the cost of another dependency.
- **`std::execution::par`** is standard C++, but actual backend behavior and deployment requirements vary by standard library.

For the DLL ABI, expose a batch function such as `ts_integrate_batch(const Params* in, size_t count, Result* out, int threads)`. Keep exceptions inside the DLL, return status codes, and make each output slot independent. Deterministic reproducibility is easiest when each integral performs its own serial node reduction and only the outer scenario loop runs in parallel.


# Part VI. Practical selection guide

| Problem shape | Preferred first choice |
|---|---|
| Endpoint singularity on finite interval | Tanh-Sinh / double exponential |
| General smooth finite interval | SciPy `quad`, Gauss-Legendre, or Clenshaw-Curtis |
| Expensive smooth vectorized integrand | Gauss-Legendre or Clenshaw-Curtis with NumPy batches |
| `exp(-x^2)` weighted real-line integral | Gauss-Hermite |
| Many independent Excel rows | Numba `prange` now; C++ outer-loop parallelism in the DLL |
| One tiny cheap integral | Lean scalar, JIT scalar, or native C++; not small-array NumPy |
| Production reference / fallback | SciPy `quad` with explicit tolerances and warnings checked |

Do not select from timing alone. Compare accuracy, failure behavior, evaluation count, setup/caching cost, and whether the integrand naturally vectorizes.


# Part VII. Golub-Welsch custom Gaussian quadrature

Golub-Welsch constructs an `n`-point Gaussian rule from the symmetric tridiagonal **Jacobi matrix** associated with orthogonal polynomials for a measure

$$
d\mu(x)=w(x)\,dx \quad\text{or}\quad d\mu(x)=\sum_j m_j\,\delta(x-x_j).
$$

If the orthonormal recurrence is

$$
x p_k(x)=\beta_{k+1}p_{k+1}(x)+\alpha_kp_k(x)+\beta_kp_{k-1}(x),
$$

then the eigenvalues of the Jacobi matrix are the abscissas and the squared first components of its normalized eigenvectors, multiplied by total mass \(\mu_0\), are the quadrature weights.

“Any shape” therefore means any positive measure for which usable recurrence coefficients, moments, or a representative positive discrete measure can be supplied. Signed or complex measures do not generally produce the standard positive-weight Gaussian rule. Raw high-order moment inversion is included for experimentation but becomes ill-conditioned quickly; the discrete Stieltjes/Lanczos route is normally safer for custom densities.


In [16]:
from scipy.linalg import eigh_tridiagonal
from scipy.stats import t as student_t, weibull_min


def golub_welsch(alpha, beta, mu0=1.0):
    """Nodes and weights from Jacobi recurrence coefficients.

    alpha: diagonal entries alpha[0:n]
    beta: positive off-diagonal entries beta[0:n-1]
    mu0: total mass of the measure
    """
    alpha=np.asarray(alpha,dtype=float)
    beta=np.asarray(beta,dtype=float)
    if alpha.ndim!=1 or beta.ndim!=1 or beta.size!=alpha.size-1:
        raise ValueError('Need len(beta) == len(alpha)-1')
    if mu0<=0 or np.any(beta<=0):
        raise ValueError('Positive measure requires mu0>0 and beta>0')
    nodes,vecs=eigh_tridiagonal(alpha,beta)
    weights=mu0*vecs[0,:]**2
    return nodes,weights


def gw_from_discrete_measure(x, mass, n, reorthogonalize=True):
    """Stieltjes/Lanczos construction for any positive discrete approximation."""
    x=np.asarray(x,dtype=float).ravel(); mass=np.asarray(mass,dtype=float).ravel()
    if x.size!=mass.size or n<1 or n>x.size: raise ValueError('invalid dimensions/order')
    if np.any(mass<0) or not np.all(np.isfinite(mass)): raise ValueError('mass must be finite and nonnegative')
    mu0=float(mass.sum())
    if mu0<=0: raise ValueError('measure has zero mass')
    q=np.sqrt(mass/mu0)
    q_prev=np.zeros_like(q); beta_prev=0.0
    basis=[]; alpha=np.empty(n); off=np.empty(max(0,n-1))
    for k in range(n):
        z=x*q-beta_prev*q_prev
        alpha[k]=np.dot(q,z); z=z-alpha[k]*q
        if reorthogonalize:
            for qi in basis:
                z -= np.dot(qi,z)*qi
        basis.append(q.copy())
        if k<n-1:
            beta_next=np.linalg.norm(z)
            if beta_next <= np.finfo(float).eps*np.linalg.norm(x):
                raise ValueError('measure supports fewer independent polynomial degrees')
            off[k]=beta_next; q_prev,q=q,z/beta_next; beta_prev=beta_next
    return golub_welsch(alpha,off,mu0)


def gw_from_density(density,a,b,n,grid_size=16385):
    """Custom positive continuous density -> fine trapezoidal discrete measure -> GW."""
    x=np.linspace(a,b,grid_size)
    rho=np.asarray(density(x),dtype=float)
    if np.any(rho<0) or np.any(~np.isfinite(rho)): raise ValueError('density must be finite and nonnegative')
    dx=(b-a)/(grid_size-1); mass=rho*dx; mass[[0,-1]]*=0.5
    return gw_from_discrete_measure(x,mass,n)


def gw_from_raw_moments(moments,n):
    """Experimental moment-to-Jacobi route via polynomial Gram-Schmidt.

    Requires moments m_0 ... m_{2n}. Scale/shift first; use modest n.
    """
    m=np.asarray(moments,dtype=float)
    if m.size<2*n+1 or m[0]<=0: raise ValueError('need positive moments m[0:2n+1]')
    def inner(c,d,shift=0):
        return sum(c[i]*d[j]*m[i+j+shift] for i in range(len(c)) for j in range(len(d)))
    polys=[]; alpha=np.empty(n); beta=np.empty(max(0,n-1))
    for k in range(n):
        c=np.zeros(k+1); c[k]=1.0
        for q in polys:
            c[:q.size] -= inner(c,q)*q
        norm=math.sqrt(inner(c,c)); c/=norm; polys.append(c)
        alpha[k]=inner(c,c,1)
        if k>0: beta[k-1]=inner(polys[k-1],c,1)
    return golub_welsch(alpha,beta,m[0])


def apply_rule(f,nodes,weights):
    return np.dot(weights,np.asarray(f(nodes),dtype=float))


## Built-in Gaussian families through Golub-Welsch

This table uses recurrence coefficients to regenerate common Gaussian rules: Legendre, Hermite, Laguerre and Jacobi. Hermite uses the physicists' weight \(e^{-x^2}\); Laguerre uses \(x^\alpha e^{-x}\); Jacobi uses \((1-x)^a(1+x)^b\).


In [17]:
def gw_legendre(n):
    k=np.arange(n,dtype=float); alpha=np.zeros(n)
    beta=k[1:]/np.sqrt(4*k[1:]**2-1)
    return golub_welsch(alpha,beta,2.0)

def gw_hermite(n):
    alpha=np.zeros(n); beta=np.sqrt(np.arange(1,n)/2)
    return golub_welsch(alpha,beta,math.sqrt(math.pi))

def gw_laguerre(n,a=0.0):
    if a<=-1: raise ValueError('a must exceed -1')
    k=np.arange(n,dtype=float); alpha=2*k+a+1
    beta=np.sqrt(np.arange(1,n)*(np.arange(1,n)+a))
    return golub_welsch(alpha,beta,float(special.gamma(a+1)))

def gw_jacobi(n,a=0.0,b=0.0):
    if a<=-1 or b<=-1: raise ValueError('a,b must exceed -1')
    k=np.arange(n,dtype=float)
    den=(2*k+a+b)*(2*k+a+b+2)
    alpha=np.divide(b*b-a*a,den,out=np.array([(b-a)/(a+b+2)]*n,dtype=float),where=den!=0)
    j=np.arange(1,n,dtype=float)
    beta=np.sqrt(4*j*(j+a)*(j+b)*(j+a+b)/((2*j+a+b)**2*((2*j+a+b)**2-1)))
    mu0=2**(a+b+1)*special.beta(a+1,b+1)
    return golub_welsch(alpha,beta,float(mu0))

family_rows=[]
for family,maker,exact,f in [
 ('Legendre',lambda:gw_legendre(16),2/7,lambda x:x**6),
 ('Hermite',lambda:gw_hermite(16),math.sqrt(math.pi)/2,lambda x:x**2),
 ('Laguerre',lambda:gw_laguerre(16),2.0,lambda x:x**2),
 ('Jacobi a=.3 b=.7',lambda:gw_jacobi(16,.3,.7),None,lambda x:np.exp(x)),
]:
    t0=perf_counter(); xn,wn=maker(); setup=perf_counter()-t0
    t0=perf_counter(); val=apply_rule(f,xn,wn); run=perf_counter()-t0
    family_rows.append((family,val,exact,setup,run,wn.sum(),xn.min(),xn.max()))
family_rows


[('Legendre',
  0.2857142857142866,
  0.2857142857142857,
  0.0078056000002106884,
  3.8800000766059384e-05,
  2.0,
  -0.9894009349916493,
  0.9894009349916499),
 ('Hermite',
  0.8862269254527595,
  0.8862269254527579,
  0.0003944000000046799,
  8.500001058564521e-06,
  1.7724538509055148,
  -4.68873893930581,
  4.688738939305819),
 ('Laguerre',
  2.000000000000002,
  2.0,
  0.00010000000111176632,
  6.199999916134402e-06,
  1.0000000000000004,
  0.08764941047892773,
  51.70116033954331),
 ('Jacobi a=.3 b=.7',
  2.093562254087823,
  None,
  0.00012820000119972974,
  8.900000466383062e-06,
  1.6309532725293923,
  -0.9798113477972169,
  0.985937599112654)]

# Part VIII. Cross-domain examples

The cases below include continuous integrals, expectations under custom distributions, discrete measures, oscillatory transforms and weighted Gaussian rules.

- **Physics**: blackbody spectral integral; quantum-harmonic-oscillator Gaussian expectation; diffraction/Fourier integral.
- **Chemistry**: Arrhenius rate averaged over a temperature distribution; Gaussian configuration integral.
- **Biology**: Hill-response population average; survival expectation under a non-standard lifetime mixture.
- **Finance**: discounted option payoff under a custom skew-Student-t/Weibull mixture; expected loss over a discrete credit migration measure.
- **Pure mathematics**: endpoint-singular beta integral and oscillatory integral.
- **Applied mathematics**: Fredholm-kernel action, convolution integral and discrete empirical expectation.

The custom “skew-t/Weibull mix” is a positive numerical density on a finite interval: a shifted/scaled Student-t component with different left/right scales, mixed with a shifted Weibull component, then normalized numerically. Golub-Welsch consumes the density without requiring a named closed-form orthogonal polynomial family.


In [18]:
RESULTS=[]
def timed_result(domain,problem,method,fn,reference=np.nan,repeats=11,setup_us=0.0,notes=''):
    t,val=median_time(fn,repeats)
    val=float(val); err=abs(val-reference) if np.isfinite(reference) else np.nan
    RESULTS.append(dict(domain=domain,problem=problem,method=method,value=val,reference=float(reference),
                        abs_error=err,median_apply_us=t*1e6,setup_us=float(setup_us),
                        first_call_us=float(setup_us+t*1e6),notes=notes))
    return val

def setup_timed(fn,repeats=5):
    samples=[]; out=None
    for _ in range(repeats):
        t0=perf_counter(); out=fn(); samples.append(perf_counter()-t0)
    return np.median(samples)*1e6,out

# Physics: Planck integral. Laguerre absorbs exp(-x).
ref_planck=math.pi**4/15
setup,xw=setup_timed(lambda:gw_laguerre(48)); xL,wL=xw
timed_result('Physics','Planck spectral integral','GW-Laguerre 48',
             lambda:apply_rule(lambda x:x**3/(-np.expm1(-x)),xL,wL),ref_planck,setup_us=setup)
timed_result('Physics','Planck spectral integral','SciPy quad',
             lambda:integrate.quad(lambda x:x**3*math.exp(-x)/(-math.expm1(-x)) if x>0 else 0.0,0,np.inf)[0],ref_planck)

# Physics: unnormalised harmonic Gaussian moment.
setup,xw=setup_timed(lambda:gw_hermite(32)); xH,wH=xw; ref_qho=math.sqrt(math.pi)/2
timed_result('Physics','QHO x^2 unnormalised','GW-Hermite 32',lambda:apply_rule(lambda x:x*x,xH,wH),ref_qho,setup_us=setup)

# Chemistry: Arrhenius average under the unnormalised Hermite measure exp(-z^2)dz.
T0,sig,EaR=600.0,25.0,8000.0
ref_arr=integrate.quad(lambda z:math.exp(-z*z-EaR/(T0+math.sqrt(2)*sig*z)),-8,8,epsabs=1e-14)[0]
timed_result('Chemistry','Arrhenius Hermite integral','GW-Hermite 32',
             lambda:apply_rule(lambda z:np.exp(-EaR/(T0+math.sqrt(2)*sig*z)),xH,wH),ref_arr,setup_us=setup,
             notes='Unnormalised; divide by sqrt(pi) for expectation')

# Biology: smooth expectation after inverse-CDF transform.
kshape,scale,K,hill=1.7,2.0,1.2,3.0
bio_f=lambda t: ((weibull_min.ppf(t,kshape,scale=scale)**hill)/(K**hill+weibull_min.ppf(t,kshape,scale=scale)**hill))
ref_bio=integrate.quad(lambda t:float(bio_f(t)),0,1,points=[0,1],limit=500,epsabs=1e-13)[0]
setup,xw=setup_timed(lambda:gw_legendre(96)); xg,wg=xw; ug=(xg+1)/2; uw=wg/2
timed_result('Biology','Hill response under Weibull','GW-Legendre 96',lambda:apply_rule(bio_f,ug,uw),ref_bio,setup_us=setup)

# Two-piece Student-t plus shifted-Weibull positive numerical mixture.
def custom_mix_raw(x):
    x=np.asarray(x); loc=-0.15; z=x-loc
    side_scale=np.where(z<0,0.55,1.25)
    two_piece_t=student_t.pdf(z/side_scale,df=5)/side_scale
    y=x+3.0
    weib=np.where(y>=0,weibull_min.pdf(y,1.4,scale=1.8),0.0)
    return 0.72*two_piece_t+0.28*weib
lo,hi=-8.0,12.0
norm=integrate.quad(lambda x:float(custom_mix_raw(x)),lo,hi,limit=1000,epsabs=1e-13)[0]
mix_pdf=lambda x:custom_mix_raw(x)/norm

# Construct both orders and retain setup separately.
setup48,xw48=setup_timed(lambda:gw_from_density(mix_pdf,lo,hi,48,grid_size=32769),repeats=3)
setup128,xw128=setup_timed(lambda:gw_from_density(mix_pdf,lo,hi,128,grid_size=65537),repeats=3)
xmix48,wmix48=xw48; xmix128,wmix128=xw128

# Finance call has a kink. Keep global 48 as requested, add global 128 convergence comparison.
S0,Kstrike,mu,sigma,disc=100.0,105.0,-0.02,0.22,math.exp(-0.04)
payoff=lambda x:disc*np.maximum(S0*np.exp(mu+sigma*x)-Kstrike,0.0)
xK=(math.log(Kstrike/S0)-mu)/sigma
ref_call=integrate.quad(lambda x:float(payoff(x)*mix_pdf(x)),xK,hi,epsabs=1e-12,limit=1000)[0]
timed_result('Finance','Call: two-piece-t/Weibull mix','Custom GW 48 global',lambda:apply_rule(payoff,xmix48,wmix48),ref_call,setup_us=setup48,notes='Global rule; kink limits convergence')
timed_result('Finance','Call: two-piece-t/Weibull mix','Custom GW 128 global',lambda:apply_rule(payoff,xmix128,wmix128),ref_call,setup_us=setup128,notes='Higher order convergence check')
timed_result('Finance','Call: two-piece-t/Weibull mix','SciPy quad split at strike',lambda:integrate.quad(lambda x:float(payoff(x)*mix_pdf(x)),xK,hi,epsabs=1e-12,limit=1000)[0],ref_call)

# Tail indicator: retain global rules, but also show the correct interval-split construction.
threshold=1.5
ref_surv=integrate.quad(lambda x:float(mix_pdf(x)),threshold,hi,epsabs=1e-13,limit=1000)[0]
timed_result('Biology','Tail probability custom mixture','Custom GW 48 global indicator',lambda:apply_rule(lambda x:(x>=threshold).astype(float),xmix48,wmix48),ref_surv,setup_us=setup48)
timed_result('Biology','Tail probability custom mixture','Custom GW 128 global indicator',lambda:apply_rule(lambda x:(x>=threshold).astype(float),xmix128,wmix128),ref_surv,setup_us=setup128)
setup_tail,xtail=setup_timed(lambda:gw_from_density(mix_pdf,threshold,hi,48,grid_size=32769),repeats=3)
xt,wt=xtail
timed_result('Biology','Tail probability custom mixture','Custom GW 48 truncated interval',lambda:wt.sum(),ref_surv,setup_us=setup_tail,notes='Rule built on [threshold,hi]')

# Pure mathematics: endpoint-singular beta integral.
aB,bB=.35,.65; ref_beta=float(special.beta(aB,bB))
timed_result('Pure math','Beta endpoint singularity','Tanh-Sinh faithful',lambda:quad_tanh_sinh_faithful(lambda x:x**(aB-1)*(1-x)**(bB-1),0,1).value,ref_beta)
timed_result('Pure math','Beta endpoint singularity','SciPy quad',lambda:integrate.quad(lambda x:x**(aB-1)*(1-x)**(bB-1),0,1,points=[0,1],epsabs=1e-13)[0],ref_beta)

# Applied mathematics: split the Fredholm kink at t=s0.
s0=.3; u=lambda x:np.sin(np.pi*x)
ref_fred=integrate.quad(lambda t:math.exp(-abs(s0-t))*math.sin(math.pi*t),0,1,points=[s0],epsabs=1e-14)[0]
xcc,wcc=clenshaw_curtis_rule(256); tcc=(xcc+1)/2; tw=wcc/2
timed_result('Applied math','Fredholm kernel action','Clenshaw-Curtis 256 global',lambda:apply_rule(lambda t:np.exp(-np.abs(s0-t))*u(t),tcc,tw),ref_fred)
def cc_interval(f,a,b,n=128): return clenshaw_curtis(f,a,b,n)
timed_result('Applied math','Fredholm kernel action','Clenshaw-Curtis split',lambda:cc_interval(lambda t:np.exp(-(s0-t))*u(t),0,s0,128)+cc_interval(lambda t:np.exp(-(t-s0))*u(t),s0,1,128),ref_fred)

# Discrete expectation: exact because all five support points are retained.
states=np.array([0.0,.01,.05,.2,.6]); probs=np.array([.90,.05,.025,.015,.01]); exposure=1_000_000.0
setup_disc,xwd=setup_timed(lambda:gw_from_discrete_measure(states,probs,5)); xdisc,wdisc=xwd
ref_loss=exposure*np.dot(probs,states)
timed_result('Finance','Discrete credit expected loss','Discrete GW exact',lambda:apply_rule(lambda x:exposure*x,xdisc,wdisc),ref_loss,setup_us=setup_disc)

# Chemistry finite smooth partition integral.
ref_part=integrate.quad(lambda x:math.exp(-(x*x+.1*x**4)),-4,4,epsabs=1e-14)[0]
setup_gl,xwgl=setup_timed(lambda:gw_legendre(96)); xgl,wgl=xwgl; xx=4*xgl; ww=4*wgl
timed_result('Chemistry','Anharmonic partition integral','GW-Legendre 96',lambda:apply_rule(lambda x:np.exp(-(x*x+.1*x**4)),xx,ww),ref_part,setup_us=setup_gl)


1.6740858555645786

# Part IX. FFT-based integration and convolution timing

For a uniformly sampled periodic function, the integral over one period is the zero-frequency Fourier coefficient multiplied by the interval length. Computing a full FFT merely to obtain that coefficient is normally slower than a direct sum; the FFT becomes useful when the spectrum itself is needed or when many circular convolutions/correlations are required.

The second example computes an entire periodic convolution. Direct quadrature at every output point is \(O(N^2)\); FFT convolution is \(O(N\log N)\).


In [19]:
# Periodic integral: direct trapezoidal sum versus FFT zero mode.
NFFT=2**16; L=2*np.pi
xf=np.arange(NFFT)*L/NFFT
ff=np.exp(np.cos(xf))*(1+0.2*np.cos(17*xf))
ref_fft=2*np.pi*(special.iv(0,1.0)+0.2*special.iv(17,1.0))  # exact Fourier/Bessel coefficient

timed_result('Applied math','Periodic integral','Direct periodic sum',lambda:(L/NFFT)*ff.sum(),ref_fft,repeats=31)
timed_result('Applied math','Periodic integral','FFT zero mode',lambda:(L/NFFT)*np.fft.rfft(ff)[0].real,ref_fft,repeats=31)

# Circular convolution h(x)=int f(t)g(x-t)dt on a periodic grid.
NC=4096; dx=2*np.pi/NC; xc=np.arange(NC)*dx
fc=np.exp(np.cos(xc))*(1+0.1*np.sin(3*xc)); gc=np.exp(.5*np.cos(xc))*(1+0.2*np.sin(5*xc))
def fft_convolution(): return dx*np.fft.irfft(np.fft.rfft(fc)*np.fft.rfft(gc),n=NC)
def direct_convolution(): return dx*np.array([np.dot(fc,np.roll(gc,i)) for i in range(NC)])  # h[i]=sum_j f[j]g[i-j]
t_fft,h_fft=median_time(fft_convolution,repeats=9)
t_dir,h_dir=median_time(direct_convolution,repeats=3)
RESULTS.append(dict(domain='Applied math',problem='Periodic convolution all points',method='FFT convolution',value=float(h_fft[0]),reference=float(h_dir[0]),abs_error=float(np.max(np.abs(h_fft-h_dir))),median_apply_us=t_fft*1e6,setup_us=0.0,first_call_us=t_fft*1e6,notes='asymmetric convention check'))
RESULTS.append(dict(domain='Applied math',problem='Periodic convolution all points',method='Direct O(N^2)',value=float(h_dir[0]),reference=float(h_dir[0]),abs_error=0.0,median_apply_us=t_dir*1e6,setup_us=0.0,first_call_us=t_dir*1e6,notes='direct reference'))
print('FFT convolution speedup:',t_dir/t_fft,'max error:',np.max(np.abs(h_fft-h_dir)))


FFT convolution speedup: 2143.87390357706 max error: 0.04265448995012555


# Part X. Complete implemented-method inventory and consolidated timings

The inventory distinguishes an implemented algorithm from a benchmark/application instance. The final results table contains all cross-domain runs from Parts VIII-IX. Earlier benchmark sections remain above for the detailed smooth/singular comparison.


In [20]:
METHOD_INVENTORY=[
 ('Adaptive Simpson','Adaptive Newton-Cotes','Finite interval','Adaptive; endpoint values required unless transformed'),
 ('Clenshaw-Curtis','Cosine/Chebyshev grid','Finite interval','Closed endpoints; cache rule; transform singular endpoints'),
 ('Gauss-Legendre','Gaussian','Finite interval, weight 1','Fixed order; excellent for smooth functions'),
 ('Gauss-Hermite','Gaussian','Real line, weight exp(-x^2)','Weighted integral'),
 ('Gauss-Laguerre','Gaussian','[0,inf), weight x^a exp(-x)','Implemented through Golub-Welsch'),
 ('Gauss-Jacobi','Gaussian','[-1,1], Jacobi weight','Implemented through Golub-Welsch'),
 ('Golub-Welsch recurrence','Custom Gaussian','Any positive measure with recurrence','Eigenvalue construction'),
 ('Golub-Welsch discrete','Custom Gaussian','Any positive discrete approximation','Stieltjes/Lanczos; supports custom density grids'),
 ('Golub-Welsch raw moments','Custom Gaussian','Positive measure with moments','Experimental; ill-conditioned at high order'),
 ('Tanh-Sinh faithful','Double exponential','Finite interval','Workbook-faithful endpoint-singularity specialist'),
 ('Tanh-Sinh NumPy','Double exponential','Finite interval','Level-batched; useful with vectorized/batched work'),
 ('Tanh-Sinh Numba','Double exponential','Finite interval','Compiled scalar and parallel batches'),
 ('SciPy quad','QUADPACK adaptive','Finite/infinite intervals','Library reference/fallback'),
 ('SciPy fixed_quad','Gaussian','Finite interval','SciPy fixed-order rule'),
 ('SciPy tanhsinh','Double exponential','Finite interval','Used when present in installed SciPy'),
 ('Periodic direct sum','Uniform discrete','Periodic sampled integral','Zero Fourier mode without FFT overhead'),
 ('FFT zero mode','Spectral/discrete','Periodic sampled integral','Useful when spectrum is also needed'),
 ('FFT convolution','Spectral/discrete','Periodic convolution','O(N log N) all-output convolution'),
]
print(f"{'Method':28s} {'Family':23s} {'Domain/weight':36s} Notes")
print('-'*130)
for row in METHOD_INVENTORY: print(f'{row[0]:28s} {row[1]:23s} {row[2]:36s} {row[3]}')


Method                       Family                  Domain/weight                        Notes
----------------------------------------------------------------------------------------------------------------------------------
Adaptive Simpson             Adaptive Newton-Cotes   Finite interval                      Adaptive; endpoint values required unless transformed
Clenshaw-Curtis              Cosine/Chebyshev grid   Finite interval                      Closed endpoints; cache rule; transform singular endpoints
Gauss-Legendre               Gaussian                Finite interval, weight 1            Fixed order; excellent for smooth functions
Gauss-Hermite                Gaussian                Real line, weight exp(-x^2)          Weighted integral
Gauss-Laguerre               Gaussian                [0,inf), weight x^a exp(-x)          Implemented through Golub-Welsch
Gauss-Jacobi                 Gaussian                [-1,1], Jacobi weight                Implemented through Golub

In [21]:
# Add the common one-dimensional comparison runs to the consolidated store.
for problem,label,val,err,us,status in all_rows:
    RESULTS.append(dict(domain='1D benchmark',problem=problem,method=label,value=val,
                        reference=val-err if np.isfinite(err) else np.nan,abs_error=err,
                        median_apply_us=us,setup_us=0.0,first_call_us=us,notes=status))

print(f"{'Domain':13s} {'Problem':36s} {'Method':31s} {'Abs error':>11s} {'Setup us':>11s} {'Apply us':>11s} {'First us':>11s}")
print('-'*140)
for r in sorted(RESULTS,key=lambda z:(z['domain'],z['problem'],z['median_apply_us'])):
    print(f"{r['domain'][:13]:13s} {r['problem'][:36]:36s} {r['method'][:31]:31s} {r['abs_error']:11.3e} {r['setup_us']:11.2f} {r['median_apply_us']:11.2f} {r['first_call_us']:11.2f}")
try:
    import pandas as pd
    timing_table=pd.DataFrame(RESULTS).sort_values(['domain','problem','median_apply_us']).reset_index(drop=True)
    display(timing_table)
except Exception:
    timing_table=RESULTS


Domain        Problem                              Method                            Abs error    Setup us    Apply us    First us
--------------------------------------------------------------------------------------------------------------------------------------------
1D benchmark  endpoint sqrt                        Adaptive Simpson transformed      1.110e-16        0.00        2.40        2.40
1D benchmark  endpoint sqrt                        Gauss-Legendre 128 cached         4.852e-08        0.00        3.40        3.40
1D benchmark  endpoint sqrt                        Clenshaw-Curtis 256               6.476e-09        0.00        3.60        3.60
1D benchmark  endpoint sqrt                        SciPy fixed_quad 128              4.852e-08        0.00       11.80       11.80
1D benchmark  endpoint sqrt                        Gauss-Legendre transformed        2.331e-15        0.00       12.50       12.50
1D benchmark  endpoint sqrt                        Clenshaw-Curtis transf

,domain,problem,method,value,reference,abs_error,median_apply_us,setup_us,first_call_us,notes
0,1D benchmark,endpoint sqrt,Adaptive Simpson transformed,6.666667e-01,0.666667,1.110223e-16,2.400000,0.000000e+00,2.400000e+00,ok
1,1D benchmark,endpoint sqrt,Gauss-Legendre 128 cached,6.666667e-01,0.666667,4.852382e-08,3.400000,0.000000e+00,3.400000e+00,ok
2,1D benchmark,endpoint sqrt,Clenshaw-Curtis 256,6.666667e-01,0.666667,6.476292e-09,3.600000,0.000000e+00,3.600000e+00,ok
3,1D benchmark,endpoint sqrt,SciPy fixed_quad 128,6.666667e-01,0.666667,4.852381e-08,11.800001,0.000000e+00,1.180000e+01,ok
4,1D benchmark,endpoint sqrt,Gauss-Legendre transformed,6.666667e-01,0.666667,2.331468e-15,12.500001,0.000000e+00,1.250000e+01,ok
5,1D benchmark,endpoint sqrt,Clenshaw-Curtis transformed,6.666667e-01,0.666667,1.110223e-16,14.000001,0.000000e+00,1.400000e+01,ok
6,1D benchmark,endpoint sqrt,Tanh-Sinh faithful,6.666667e-01,0.666667,2.220446e-16,28.200000,0.000000e+00,2.820000e+01,ok
7,1D benchmark,endpoint sqrt,SciPy quad,6.666667e-01,0.666667,0.000000e+00,176.800000,0.000000e+00,1.768000e+02,ok
8,1D benchmark,endpoint sqrt,Tanh-Sinh NumPy,6.666667e-01,0.666667,4.440892e-16,244.400000,0.000000e+00,2.444000e+02,ok
9,1D benchmark,log singular,Gauss-Legendre 128 cached,-9.999618e-01,-1.000000,3.823763e-05,5.000000,0.000000e+00,5.000000e+00,ok


# Part XI. Corrections, invariants and aligned interpretation

The audit distinguishes rule construction, repeated application and first-use cost:

$$
T_{\mathrm{first}}=T_{\mathrm{setup}}+T_{\mathrm{apply}}.
$$

For $M$ applications of the same precomputed rule,

$$
T_{\mathrm{amortized}}
=
\frac{T_{\mathrm{setup}}}{M}+T_{\mathrm{apply}}.
$$

A precomputed Gaussian rule should therefore be compared with adaptive integration using repeated-use cost only when reuse is intended. Otherwise, first-use cost is the fairer comparison.

No single quadrature family applies naturally to every problem. The aligned suite uses each implemented family on its defining measure or after an explicit transformation. Incompatible combinations are identified rather than forced.


In [22]:
# Core invariants and independent rule equivalence checks.
def assert_rule(nodes,weights,mu0,tol=5e-12):
    assert np.all(np.isfinite(nodes)) and np.all(np.isfinite(weights))
    assert np.min(weights) > -1e-15  # tiny signed roundoff is tolerated at extreme nodes
    assert abs(weights.sum()-mu0) <= tol*max(1,abs(mu0))
    assert np.all(np.diff(nodes)>0)

for n in (4,8,16,32,64):
    x,w=gw_legendre(n); assert_rule(x,w,2.0)
    xl,wl=leggauss(n)
    assert np.max(np.abs(x-xl))<2e-13 and np.max(np.abs(w-wl))<2e-13
    x,w=gw_hermite(n); assert_rule(x,w,math.sqrt(math.pi))
    xh,wh=hermgauss(n)
    assert np.max(np.abs(x-xh))<5e-13 and np.max(np.abs(w-wh))<5e-13

# Polynomial exactness: n-point Gaussian rule integrates through degree 2n-1.
for maker,mu in [(gw_legendre,lambda k:0 if k%2 else 2/(k+1)),
                 (gw_hermite,lambda k:0 if k%2 else special.gamma((k+1)/2))]:
    n=8; x,w=maker(n)
    for k in range(2*n):
        assert abs(np.dot(w,x**k)-mu(k)) < 2e-10*max(1,abs(mu(k)))
print('Golub-Welsch invariants, NumPy rule equivalence and polynomial exactness: PASS')


Golub-Welsch invariants, NumPy rule equivalence and polynomial exactness: PASS


## All-method applicability and aligned one-dimensional benchmark

- Adaptive Simpson, Clenshaw-Curtis, Gauss-Legendre, Tanh-Sinh and SciPy `quad` are compared directly on finite-interval problems.
- Gauss-Hermite, Laguerre and Jacobi are evaluated on their defining weighted measures.
- FFT is evaluated on uniform periodic samples, where it is mathematically natural.
- Golub-Welsch supplies all Gaussian nodes and weights, including custom measures.

This is alignment, not artificial coercion. Gauss-Hermite is not treated as a finite-interval rule, and FFT is not presented as an adaptive singular-integral solver.


In [23]:
ALIGNED=[]
def add_aligned(problem,method,value,reference,seconds,applicable=True,note=''):
    ALIGNED.append(dict(problem=problem,method=method,value=float(value) if applicable else np.nan,
                        reference=float(reference),abs_error=abs(float(value)-reference) if applicable else np.nan,
                        median_us=seconds*1e6 if applicable else np.nan,applicable=applicable,note=note))

# Smooth finite interval: every general finite rule.
fv=lambda x:np.exp(-x*x)*np.cos(7*x); fs=lambda x:math.exp(-x*x)*math.cos(7*x)
a,b=-3.,3.; ref=reference_value(fv,a,b)
finite_methods={
 'Adaptive Simpson':lambda:adaptive_simpson(fs,a,b,1e-12)[0],
 'Clenshaw-Curtis 256':lambda:clenshaw_curtis(fv,a,b,256),
 'GW-Legendre 128':lambda:gauss_legendre_cached(fv,a,b,128),
 'Tanh-Sinh faithful':lambda:quad_tanh_sinh_faithful(fs,a,b).value,
 'Tanh-Sinh NumPy':lambda:quad_tanh_sinh_vectorized(fv,a,b).value,
 'SciPy quad':lambda:integrate.quad(fs,a,b,epsabs=1e-13)[0],
 'SciPy fixed_quad':lambda:integrate.fixed_quad(fv,a,b,n=128)[0]}
for name,fn in finite_methods.items():
    t,v=median_time(fn,21); add_aligned('Smooth finite interval',name,v,ref,t)

# Weighted canonical problems.
for problem,method,fn,ref in [
 ('Hermite weighted','GW-Hermite 32',lambda:apply_rule(lambda x:np.cos(3*x),*gw_hermite(32)),math.sqrt(math.pi)*math.exp(-9/4)),
 ('Laguerre weighted','GW-Laguerre 32',lambda:apply_rule(lambda x:x**3,*gw_laguerre(32)),6.0),
 ('Jacobi weighted','GW-Jacobi 32',lambda:apply_rule(lambda x:x**4,*gw_jacobi(32,.3,.7)),
  integrate.quad(lambda x:x**4*(1-x)**.3*(1+x)**.7,-1,1,epsabs=1e-14)[0])]:
    t,v=median_time(fn,21); add_aligned(problem,method,v,ref,t)

# Periodic sampled integral: direct zero mode and FFT zero mode.
t,v=median_time(lambda:(L/NFFT)*ff.sum(),21); add_aligned('Periodic sampled integral','Direct periodic sum',v,ref_fft,t)
t,v=median_time(lambda:(L/NFFT)*np.fft.rfft(ff)[0].real,21); add_aligned('Periodic sampled integral','FFT zero mode',v,ref_fft,t)

try:
    aligned_table=pd.DataFrame(ALIGNED).sort_values(['problem','median_us'])
    display(aligned_table)
except Exception: aligned_table=ALIGNED


,problem,method,value,reference,abs_error,median_us,applicable,note
7,Hermite weighted,GW-Hermite 32,0.186815,0.186815,2.914335e-15,75.000000,True,
9,Jacobi weighted,GW-Jacobi 32,0.222832,0.222832,3.486100e-14,102.000000,True,
8,Laguerre weighted,GW-Laguerre 32,6.000000,6.000000,7.105427e-15,80.899999,True,
10,Periodic sampled integral,Direct periodic sum,7.954927,7.954927,1.776357e-15,14.699999,True,
11,Periodic sampled integral,FFT zero mode,7.954927,7.954927,1.776357e-15,488.300000,True,
2,Smooth finite interval,GW-Legendre 128,0.000035,0.000035,2.995109e-17,7.499999,True,
1,Smooth finite interval,Clenshaw-Curtis 256,0.000035,0.000035,2.469270e-17,9.400001,True,
6,Smooth finite interval,SciPy fixed_quad,0.000035,0.000035,6.179681e-16,13.600000,True,
5,Smooth finite interval,SciPy quad,0.000035,0.000035,4.743385e-18,52.899999,True,
3,Smooth finite interval,Tanh-Sinh faithful,0.000035,0.000035,6.316968e-16,113.200000,True,


# Part XII. Introduction to cubature: 2D, 3D and 4D

Cubature extends quadrature to multiple dimensions. Tensor products preserve the one-dimensional rule's accuracy but require \(n^d\) evaluations. This is manageable in 2D and sometimes 3D, but grows rapidly in 4D and beyond.

The examples use the separable reference

$$
I_d=\int_{[-1,1]^d}\exp\!\left(\sum_{j=1}^d x_j\right)dx=(2\sinh 1)^d.
$$

Implemented adaptations:

- tensor-product Golub-Welsch Gauss-Legendre;
- tensor-product Clenshaw-Curtis;
- iterated adaptive SciPy integration for 2D and 3D checks;
- Smolyak sparse cubature using nested Clenshaw-Curtis rules.


In [24]:
from itertools import product
from math import comb

def tensor_cubature(func, rules):
    """Vectorized tensor-product cubature. rules is [(nodes,weights), ...]."""
    grids=np.meshgrid(*[r[0] for r in rules],indexing='ij')
    wg=np.meshgrid(*[r[1] for r in rules],indexing='ij')
    pts=np.stack([g.ravel() for g in grids],axis=1)
    weights=np.prod(np.stack(wg,axis=0),axis=0).ravel()
    vals=np.asarray(func(pts),dtype=float).reshape(-1)
    return np.dot(weights,vals),pts.shape[0]

def unit_legendre_rule(n): return gw_legendre(n)
def unit_cc_rule(n_intervals): return clenshaw_curtis_rule(n_intervals)
def exp_sum(pts): return np.exp(np.sum(pts,axis=1))

CUBATURE=[]
def record_cubature(dim,method,level_or_n,value,reference,evals,seconds):
    CUBATURE.append(dict(dim=dim,method=method,level_or_n=level_or_n,value=value,
                         reference=reference,abs_error=abs(value-reference),evaluations=evals,median_us=seconds*1e6))

for d in (2,3,4):
    ref=(2*math.sinh(1))**d
    for n in (4,8,12):
        rule=unit_legendre_rule(n)
        t,out=median_time(lambda d=d,rule=rule:tensor_cubature(exp_sum,[rule]*d),7)
        record_cubature(d,'Tensor GW-Legendre',n,out[0],ref,out[1],t)
    for n in (4,8,16):
        rule=unit_cc_rule(n)
        t,out=median_time(lambda d=d,rule=rule:tensor_cubature(exp_sum,[rule]*d),7)
        record_cubature(d,'Tensor Clenshaw-Curtis',n,out[0],ref,out[1],t)

# Independent adaptive checks where cost remains moderate.
ref2=(2*math.sinh(1))**2
t,v=median_time(lambda:integrate.dblquad(lambda y,x:math.exp(x+y),-1,1,lambda x:-1,lambda x:1)[0],5)
record_cubature(2,'SciPy dblquad','adaptive',v,ref2,np.nan,t)
ref3=(2*math.sinh(1))**3
t,v=median_time(lambda:integrate.tplquad(lambda z,y,x:math.exp(x+y+z),-1,1,lambda x:-1,lambda x:1,lambda x,y:-1,lambda x,y:1)[0],3)
record_cubature(3,'SciPy tplquad','adaptive',v,ref3,np.nan,t)


## Smolyak sparse cubature

A Smolyak construction combines tensor products of nested one-dimensional rules. The implementation below consolidates duplicate nodes, so `evaluations` is the number of unique function points. Increasing the sparse level raises the point count and should reduce error for smooth integrands.

Sparse grids are especially effective when the integrand is smooth and has limited high-order interaction among dimensions. They are less effective for discontinuities, sharply localized ridges or severe anisotropy unless the rule is adapted.


In [25]:
@lru_cache(maxsize=64)
def cc_nested_level(level):
    if level<1: raise ValueError('level >= 1')
    if level==1: return np.array([0.0]),np.array([2.0])
    return clenshaw_curtis_rule(2**(level-1))

def compositions(total,d,minpart=1):
    if d==1:
        if total>=minpart: yield (total,)
        return
    for first in range(minpart,total-minpart*(d-1)+1):
        for rest in compositions(total-first,d-1,minpart): yield (first,)+rest

def smolyak_rule(dim,level,round_digits=15):
    """Isotropic Smolyak A(q,d), q=level+dim-1, nested Clenshaw-Curtis."""
    if dim<1 or level<1: raise ValueError('dim, level >= 1')
    q=level+dim-1; acc={}
    for total in range(q-d+1,q+1):
        coeff=(-1)**(q-total)*comb(dim-1,q-total)
        for idx in compositions(total,dim):
            rules=[cc_nested_level(i) for i in idx]
            for choices in product(*[range(len(r[0])) for r in rules]):
                point=tuple(float(rules[j][0][choices[j]]) for j in range(dim))
                key=tuple(round(x,round_digits) for x in point)
                weight=coeff*np.prod([rules[j][1][choices[j]] for j in range(dim)])
                acc[key]=acc.get(key,0.0)+weight
    keys=[k for k,v in acc.items() if abs(v)>1e-15]
    pts=np.array(keys,dtype=float); w=np.array([acc[k] for k in keys],dtype=float)
    return pts,w

def sparse_cubature(func,dim,level):
    pts,w=smolyak_rule(dim,level)
    return np.dot(w,np.asarray(func(pts),dtype=float)),len(w)

for d in (2,3,4):
    ref=(2*math.sinh(1))**d
    for level in (2,3,4,5):
        t,out=median_time(lambda d=d,level=level:sparse_cubature(exp_sum,d,level),5)
        record_cubature(d,'Smolyak CC sparse',level,out[0],ref,out[1],t)

print(f"{'Dim':>3s} {'Method':24s} {'Level/n':>8s} {'Evals':>9s} {'Abs error':>12s} {'Median us':>12s}")
print('-'*80)
for r in sorted(CUBATURE,key=lambda z:(z['dim'],z['method'],str(z['level_or_n']))):
    ev='adaptive' if not np.isfinite(r['evaluations']) else str(int(r['evaluations']))
    print(f"{r['dim']:3d} {r['method']:24s} {str(r['level_or_n']):>8s} {ev:>9s} {r['abs_error']:12.3e} {r['median_us']:12.2f}")
try:
    cubature_table=pd.DataFrame(CUBATURE).sort_values(['dim','method','evaluations']).reset_index(drop=True)
    display(cubature_table)
except Exception: cubature_table=CUBATURE


Dim Method                    Level/n     Evals    Abs error    Median us
--------------------------------------------------------------------------------
  2 SciPy dblquad            adaptive  adaptive    8.882e-16       124.90
  2 Smolyak CC sparse               2         5    7.618e-02        65.40
  2 Smolyak CC sparse               3        13    8.193e-03       192.40
  2 Smolyak CC sparse               4        29    1.553e-04       458.70
  2 Smolyak CC sparse               5        65    6.301e-07      1107.60
  2 Tensor Clenshaw-Curtis         16       289    2.665e-15        42.50
  2 Tensor Clenshaw-Curtis          4        25    1.270e-04        39.20
  2 Tensor Clenshaw-Curtis          8        81    9.620e-11        53.20
  2 Tensor GW-Legendre             12       144    2.665e-15        50.70
  2 Tensor GW-Legendre              4        16    1.387e-06        95.00
  2 Tensor GW-Legendre              8        64    4.441e-15        63.20
  3 SciPy tplquad            ad

,dim,method,level_or_n,value,reference,abs_error,evaluations,median_us
0,2,SciPy dblquad,adaptive,5.524391,5.524391,8.881784e-16,NaN,124.900000
1,2,Smolyak CC sparse,2,5.448215,5.524391,7.617636e-02,5.0,65.400000
2,2,Smolyak CC sparse,3,5.532584,5.524391,8.193048e-03,13.0,192.400001
3,2,Smolyak CC sparse,4,5.524236,5.524391,1.553129e-04,29.0,458.700000
4,2,Smolyak CC sparse,5,5.524392,5.524391,6.301300e-07,65.0,1107.600001
5,2,Tensor Clenshaw-Curtis,4,5.524264,5.524391,1.269697e-04,25.0,39.200000
6,2,Tensor Clenshaw-Curtis,8,5.524391,5.524391,9.619683e-11,81.0,53.199999
7,2,Tensor Clenshaw-Curtis,16,5.524391,5.524391,2.664535e-15,289.0,42.500000
8,2,Tensor GW-Legendre,4,5.524390,5.524391,1.387354e-06,16.0,94.999999
9,2,Tensor GW-Legendre,8,5.524391,5.524391,4.440892e-15,64.0,63.199999


## Cubature interpretation

- Tensor Gauss-Legendre converges very rapidly for this analytic example, but point counts scale as \(n^d\).
- Tensor Clenshaw-Curtis also converges rapidly and offers nested nodes when the order doubles.
- Smolyak uses substantially fewer points at comparable one-dimensional levels. Its convergence table must be read using both error and unique evaluation count.
- For repeated ExcelBridge workloads, precompute tensor or sparse nodes and weights once, then apply many scenario functions to the same point matrix.
- For non-smooth multidimensional payoffs, split along known exercise boundaries where possible; otherwise consider adaptive cubature, quasi-Monte Carlo using `scipy.stats.qmc.Sobol`, or randomized lattice rules.
